In [171]:
### Cross-tab: TI (rows) vs TS (columns), 1990-2020 ###
import csv
import numpy as np, rasterio
from rasterio.windows import Window

TI = r"./data/5_Typology/TimeInterval-1990-2020.tif"
TS = r"./data/5_Typology/combined_rc_timeseries/combined_rc.tif"
OUT_CSV = r"./data/5_Typology/crosstab_TI_vs_TS_1990_2020.csv"
PX_M2 = 1016.79
ROWS = 128
NAMES = ["Attrition", "Aggregation", "Shrinkage", "Perforation", "Deformation",
         "Persistent", "Enlargement", "Dissection", "Fragmentation per se", "Creation"]

M = np.zeros((11, 11), np.int64)      # rows = TI class, cols = TS class, index 0 = NoData
with rasterio.open(TI) as a, rasterio.open(TS) as b:
    assert (a.width, a.height) == (b.width, b.height), "rasters are different sizes"
    for r in range(0, a.height, ROWS):
        w = Window(0, r, a.width, min(ROWS, a.height - r))
        x = a.read(1, window=w, masked=True); y = b.read(1, window=w, masked=True)
        xs = np.where(np.ma.getmaskarray(x), 0, np.ma.getdata(x)).astype(np.int64)
        ys = np.where(np.ma.getmaskarray(y), 0, np.ma.getdata(y)).astype(np.int64)
        ok = (xs <= 10) & (ys <= 10)
        M += np.bincount((xs[ok] * 11 + ys[ok]), minlength=121).reshape(11, 11)

both = M[1:, 1:].sum()
agree = np.trace(M[1:, 1:])
print(f"both classified: {both:,} px ({both * PX_M2 / 1e6:,.0f} km2)")
print(f"same class     : {agree:,} px ({agree * PX_M2 / 1e6:,.0f} km2) = {100 * agree / both:.2f}%")
print(f"TI-only (TS NoData): {M[1:, 0].sum():,} px")
print(f"TS-only (TI NoData): {M[0, 1:].sum():,} px")

print()
print("Pixels (rows = TI, cols = TS):")
print(f"{'TI_row \\ TS_col':20}" + "".join(f"{n[:6]:>8}" for n in NAMES) + f"{'TI tot':>12}")
for i in range(1, 11):
    print(f"{NAMES[i - 1]:20}" + "".join(f"{M[i, j]:>8,}" for j in range(1, 11)) + f"{M[i, 1:].sum():>12,}")
print(f"{'TS tot':20}" + "".join(f"{M[1:, j].sum():>8,}" for j in range(1, 11)) + f"{both:>12,}")

with open(OUT_CSV, "w", newline="") as f:
    wri = csv.writer(f)
    wri.writerow(["TI_row \\ TS_col"] + NAMES + ["TI_total_px", "TI_total_km2"])
    for i in range(1, 11):
        wri.writerow([NAMES[i - 1]] + [int(M[i, j]) for j in range(1, 11)]
                     + [int(M[i, 1:].sum()), round(M[i, 1:].sum() * PX_M2 / 1e6, 1)])
    wri.writerow(["TS_total_px"] + [int(M[1:, j].sum()) for j in range(1, 11)] + [int(both), ""])
    wri.writerow(["TS_total_km2"] + [round(M[1:, j].sum() * PX_M2 / 1e6, 1) for j in range(1, 11)] + ["", round(both * PX_M2 / 1e6, 1)])
print()
print("saved", OUT_CSV)


both classified: 996,757,493 px (1,013,493 km2)
same class     : 566,344,299 px (575,853 km2) = 56.82%
TI-only (TS NoData): 22,324,406 px
TS-only (TI NoData): 3,489,068 px

Pixels (rows = TI, cols = TS):
TI_row \ TS_col       Attrit  Aggreg  Shrink  Perfor  Deform  Persis  Enlarg  Dissec  Fragme  Creati      TI tot
Attrition           107,892,44933,228,37115,592,8951,721,5903,512,01614,159,3891,743,2382,412,1641,523,076 772,590 182,557,778
Aggregation         3,132,005102,742,057 640,603 138,0813,603,7025,026,30420,355,942 200,031 893,5062,925,890 139,658,121
Shrinkage           13,504,7084,641,75925,811,895 276,1434,571,74710,560,6601,060,8365,067,5472,985,168 765,273  69,245,736
Perforation         2,590,0461,902,885 745,63710,086,5872,735,9134,181,459 623,9702,080,771 767,129 153,514  25,867,911
Deformation           95,674 156,806 126,284  45,796 670,535 780,002 177,147  15,188  75,199  50,688   2,193,319
Persistent           374,107 334,708 449,615  43,829 443,69523,288,151 323,54